# 🎬 DubbingStudioX — Doblaje automático con IA en Google Colab

Evolución de SubtitleStudioX. Ahora con clonación de voz y doblaje a español latino.

---

## 🚀 Uso rápido
1. Ejecuta la **Celda 1** (tarda unos minutos).
2. Ejecuta la **Celda 2** y elige entre procesar desde cero o solo doblaje.
3. Sube tus archivos cuando se te pida y espera el resultado.

In [ ]:
# ═══════════════════════════════════════
# CELDA 1 – VERSIONES COMPATIBLES (scipy 1.14.1 + numpy moderno)
# ═══════════════════════════════════════
import os
print("✅ Clonando DubbingStudioX...")
print()
!rm -rf DubbingStudioX
!git clone https://github.com/KingEdhard/DubbingStudioX.git
%cd DubbingStudioX
print()

print("✅ Instalando dependencias. Tada de 10 a 15 minutos..")
print()
# 1. Limpiar e instalar la pareja numpy+scipy que no falla
!pip uninstall -y numpy scipy
!pip install "numpy>=2.1.0" scipy==1.14.1 --quiet
print()

print("✅ numpy y scipy compatibles instalados. Instalando resto...")
# 2. Resto de dependencias
!pip install faster-whisper sentencepiece tqdm pydub soundfile sacremoses --quiet
!pip install git+https://github.com/m-bain/whisperx.git --quiet
!pip install transformers --quiet
!pip install coqui-tts --quiet
print()

# 3. FFmpeg
!apt-get install ffmpeg -y > /dev/null
!mkdir -p bin
!ln -sf $(which ffmpeg) bin/ffmpeg.exe
!ln -sf $(which ffprobe) bin/ffprobe.exe

print("✅ Entorno listo (scipy 1.14.1). Ya puedes ejecutar la Celda 2.")

In [ ]:
print("🔍 Verificación de módulos...")

try:
    import torch
    print(f"CUDA disponible: {torch.cuda.is_available()}")

    from src.components.doblaje import generar_doblaje
    from src.components.extraccion import extraer_audio_mejorado
    from src.components.traduccion import traducir_srt
    from src.components.muxer import incrustar_subtitulos
    print("✅ Todos los módulos importados correctamente.")
except Exception as e:
    print(f"❌ Error en la verificación: {e}")
    print("⚠️ Revisa la instalación de numpy/scipy e inténtalo de nuevo.")

In [ ]:
# ═══════════════════════════════════════
# CELDA 2 – MENÚ Y PIPELINE DE DOBLAJE (con try/except)
# ═══════════════════════════════════════
import os, json, shutil, sys, subprocess
from google.colab import files
from IPython.display import clear_output
import torch

# Intentar importar los módulos del proyecto
try:
    from src.components.extraccion import extraer_audio_mejorado
    from src.components.traduccion import traducir_srt
    from src.components.muxer import incrustar_subtitulos
    from src.components.doblaje import generar_doblaje
except Exception as e:
    print(f"❌ Error importando módulos del proyecto: {e}")
    print("Asegúrate de haber ejecutado la Celda 1 correctamente.")
    raise

def transcribir_con_whisperx(wav_path):
    import whisperx
    device, compute_type = "cuda", "float16"
    print("🚀 Iniciando WhisperX Large-v3...")
    model = whisperx.load_model("large-v3", device, compute_type=compute_type, language="en")
    audio = whisperx.load_audio(wav_path)
    result = model.transcribe(audio, batch_size=16)
    print("🎯 Alineación fonética milimétrica (Wav2Vec2)...")
    model_a, metadata = whisperx.load_align_model(language_code="en", device=device)
    result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)
    
    # Guardar SRT en inglés
    srt_path = wav_path.replace("_dialogos_mejorados.wav", "_en.srt")
    def format_time(t):
        h, m, s, ms = int(t // 3600), int((t % 3600) // 60), int(t % 60), int((t % 1) * 1000)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    with open(srt_path, "w", encoding="utf-8") as f:
        for idx, seg in enumerate(result["segments"], 1):
            f.write(f"{idx}\n{format_time(seg['start'])} --> {format_time(seg['end'])}\n{seg['text'].strip()}\n\n")
    
    # Guardar JSON con segmentos (para el doblaje)
    json_path = wav_path.replace("_dialogos_mejorados.wav", "_segments.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump([{"start": s["start"], "end": s["end"], "text": s["text"].strip()} for s in result["segments"]], f, ensure_ascii=False, indent=2)
    print(f"✔ Segmentos guardados en {json_path}")
    return srt_path, json_path

# ---------- MENÚ PRINCIPAL ----------
print("🎤 DUBBING STUDIO X – Doblaje automático con voz clonada\n")
print("Elige el modo de trabajo:")
print("1. Procesar desde cero (vídeo → doblaje completo)")
print("2. Solo doblaje (ya tienes los segmentos traducidos)")
modo = input("Tu opción (1/2): ").strip()
while modo not in ['1', '2']:
    modo = input("Opción inválida. Elige 1 o 2: ").strip()

try:
    if modo == '1':
        # --- MODO 1: PROCESO COMPLETO ---
        while True:
            try:
                print("\n📤 Sube un archivo de vídeo (mp4, mkv, avi, mov...)")
                uploaded = files.upload()
                if not uploaded:
                    print("No se subió nada. ¿Salir? (S/n)")
                    if input().lower() == 's':
                        break
                    continue
                
                video_nombre = list(uploaded.keys())[0]
                print(f"\n🎬 Procesando: {video_nombre}")
                
                # 1. Extraer audio mejorado
                wav = extraer_audio_mejorado(video_nombre)
                if not wav:
                    continue
                
                # 2. Transcripción con WhisperX
                srt_ingles, json_seg = transcribir_con_whisperx(wav)
                if not srt_ingles or not json_seg:
                    continue
                
                # 3. Traducción a español (Helsinki)
                print("\n🌎 Traduciendo a español latino...")
                srt_espanol = traducir_srt(srt_ingles)
                if not srt_espanol:
                    print("⚠ Falló la traducción. Se usará el SRT en inglés como base.")
                    srt_espanol = srt_ingles
                
                # 4. Traducir los segmentos del JSON (necesario para el doblaje)
                print("🔧 Preparando segmentos para doblaje...")
                with open(json_seg, 'r') as f:
                    seg_en = json.load(f)
                from transformers import MarianMTModel, MarianTokenizer
                tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-es")
                model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-es").to(torch.device('cpu'))
                for seg in seg_en:
                    inputs = tokenizer(seg['text'], return_tensors='pt', padding=True, truncation=True, max_length=512)
                    outputs = model.generate(**inputs)
                    seg['text'] = tokenizer.decode(outputs[0], skip_special_tokens=True)
                json_es = video_nombre.rsplit('.',1)[0] + '_segments_es.json'
                with open(json_es, 'w', encoding='utf-8') as f:
                    json.dump(seg_en, f, ensure_ascii=False, indent=2)
                print(f"✔ Segmentos traducidos guardados en {json_es}")
                
                # 5. Generar audio doblado con clonación de voz
                print("\n🎙️ Generando doblaje con voz clonada (XTTS-v2 GPU)...")
                audio_doblado = generar_doblaje(json_es, wav, "doblaje_temp.wav")
                
                # 6. Multiplexar con FFmpeg
                print("\n🎬 Incrustando audio doblado y subtítulos...")
                ruta_final = incrustar_subtitulos(
                    video_nombre,
                    srt_ingles,
                    srt_espanol,
                    formato_salida=None,
                    audio_doblaje=audio_doblado
                )
                if ruta_final:
                    print(f"\n🏁 ¡Vídeo doblado listo!: {ruta_final}")
                    files.download(ruta_final)
                
                # Limpiar temporales
                for tmp in [wav, audio_doblado]:
                    if os.path.exists(tmp):
                        os.remove(tmp)
            except Exception as e:
                print(f"\n❌ Error procesando el vídeo: {e}")
            finally:
                if input("\n¿Procesar otro vídeo? (S/n): ").lower() == 'n':
                    break
    else:
        # --- MODO 2: SOLO DOBLAJE ---
        while True:
            try:
                print("\n📂 Sube el archivo JSON con los segmentos traducidos (ej: *_segments_es.json)")
                uploaded_json = files.upload()
                if not uploaded_json:
                    print("No se subió nada. ¿Salir? (S/n)")
                    if input().lower() == 's':
                        break
                    continue
                json_path = list(uploaded_json.keys())[0]
                
                print("\n🎵 Sube el audio de referencia (el audio original extraído, o un fragmento WAV)")
                uploaded_audio = files.upload()
                if not uploaded_audio:
                    print("Falta el audio. Cancelando.")
                    continue
                ref_audio = list(uploaded_audio.keys())[0]
                
                print("\n🎥 Sube el vídeo original (para incrustar el audio doblado)")
                uploaded_video = files.upload()
                if not uploaded_video:
                    print("Falta el vídeo. Cancelando.")
                    continue
                video_orig = list(uploaded_video.keys())[0]
                
                print("\n🎙️ Iniciando doblaje...")
                audio_doblado = generar_doblaje(json_path, ref_audio, "doblaje_temp.wav")
                
                print("\n🎬 Multiplexando...")
                ruta_final = incrustar_subtitulos(video_orig, None, None, audio_doblaje=audio_doblado)
                if ruta_final:
                    print(f"\n🏁 ¡Vídeo doblado listo!: {ruta_final}")
                    files.download(ruta_final)
                
                for tmp in [audio_doblado]:
                    if os.path.exists(tmp):
                        os.remove(tmp)
            except Exception as e:
                print(f"\n❌ Error en el proceso de doblaje: {e}")
            finally:
                if input("\n¿Doblar otro? (S/n): ").lower() == 'n':
                    break
except KeyboardInterrupt:
    print("\n⏹️ Proceso interrumpido por el usuario.")
except Exception as e:
    print(f"\n❌ Error inesperado: {e}")
finally:
    print("🎉 ¡Gracias por usar DubbingStudioX!")